# Two domains, one model, no rehearsal

This notebook adapts **Qwen3-0.6B** to one domain, then to a second one, and measures at every
step what was learned and what was kept. Between the two domains nothing from the first is
stored, replayed or revisited: the only thing that carries the first domain forward is a
distribution of hidden states.

It is a demonstration of Layerwise Function Anchoring (LFA), and the point of it is the
**control**. Two of the runs below have the anchor switched off — same corpus, same optimizer
steps, same everything else — and the difference between the anchored and the unanchored
column is what the method bought. If the anchor did nothing, the two columns would agree.
Two further runs then check the controls themselves, by giving each its own best number
of epochs rather than the anchored run's.

## What happens

| | |
|---|---|
| 1 | Fetch the published `p(h)` artifact and look inside it |
| 2 | Download two public-domain books, one per domain |
| 3 | Make a workspace |
| 4 | **Stage 1** — adapt to domain A, and read both axes |
| 5 | **The control** — the same stage with the anchor off |
| 6 | `extend` — fold A into the model *and* into `p(h)` |
| 7 | **Stage 2** — adapt to domain B, anchored against the extended `p(h)` |
| 8 | **The control again** — stage 2 with the anchor off, and what happens to A |
| 8b | **A check on the controls themselves** — both, at their own best dose |
| 9 | Export a plain checkpoint and generate from it |

## What you need

* One CUDA card with about 12 GB free. Everything below runs on a single device.
* `pip install lfa-anchoring` (this notebook was run against version 0.1.0).
* Network, for the artifact, the model, the two books and WikiText-2.
* **No API keys.** Every number here is a perplexity computed on your own machine; nothing in
  this package calls out to a model service.

## What it costs, and what the real recipe costs

The settings in the next cell are **demo settings**, chosen so the whole notebook finishes in
well under an hour on one RTX 3090. They are not the recommended operating point.

| | this notebook | the shipped recipe (`lfa/recipes/qwen3-0.6b.yaml`) |
|---|---|---|
| text per domain | 0.97 MB and 0.48 MB | 6.6 MB |
| epochs | 4 | 15 |
| wall clock per training run | 2-4 min | ~1 h 48 m |
| training runs | 6 (two stages, two controls, two dose checks) | 1 per stage |
| whole notebook | 23 min | — |

Everything else — rank 32, λ = 100,000, μ = 0.05, the layer schedule, the optimizer — is the
shipped recipe untouched, because λ is coupled to the rank and to the artifact and porting it
is a different regularizer (`docs/concepts.md`). What this notebook changes is corpus size and
run length, and it says so at every number it prints.

One consequence worth stating up front: this is **one run per cell, one seed, no error bars**.
It shows a mechanism at a scale you can watch. It is not a measurement of the method, and the
paper's numbers are not reproduced here.

## A note on these outputs: the loader they were produced on

These outputs were recorded on 2026-09-08 on a **fixed** corpus loader. They replace an earlier
set that a defect had quietly shaped, and the difference is worth reading before the numbers.

The per-epoch chunk offset used to be where each document *started*, so from the second epoch on
every document lost its first `offset` tokens. This notebook's corpora are exactly the shape that
hurts — each book is cut into ~3,500-character documents, a little over one 512-token chunk each —
and the old outputs showed it plainly: stage 1 trained on 164,282 tokens in epoch 1 but 92,367 in
epoch 3, about **83 %** of the token exposure it had asked for. The offset now rotates the chunk
*boundaries* instead: the leading segment becomes a chunk of its own, so every token is trained on
in every epoch. In the epoch lines below the token count is flat (164,282, then 164,281 …) while
the chunk count rises from 390 to 556 — one extra chunk per multi-chunk document.

Nothing downstream was carried over. The epoch count was re-derived from a fresh 8-epoch pass and
**moved from 5 to 4** (section 4); each control's own best dose was re-derived from its own curve
and one of the two moved as well, stage 2 from 2 epochs to 3 (section 8b).

The anchored-versus-control conclusion is unchanged in direction and **smaller in size**: the
anchor now buys +55 % on the general axis rather than the +127 % the old outputs showed, almost
all of which is the control training for one epoch fewer rather than the loader itself — at a
matched 2 epochs the stage-1 control lands in the same place under either loader (WikiText-2 18.87
now against 19.14 then). Section 8b is where that is visible.

The old stream is not available as an option: a switch that discards a prefix of every document
has no use to anyone training on their own corpus. See [`docs/verification.md`](../docs/verification.md).

Two cosmetic edits to the recorded output. The filesystem paths are rewritten to a neutral `/tmp/lfa-nb-run/...` so the notebook does not carry one machine's directory layout, and the loader's one-line notice is the wording the package prints today rather than the wording it printed on the day of the run. No number, count, or measurement is altered.


In [1]:
from pathlib import Path

# --- demo scale (see the table above) ---------------------------------------------------------
EPOCHS = 4              # the shipped recipe's 15 over-trains a corpus this small; see cell "dose"
N_WINDOWS = 200         # WikiText-2 sliding windows for the general axis (200 x 512 tokens)
EXTEND_NEED = 20_000    # hidden states collected per site when folding domain A into p(h)
EXTEND_K = 4            # mixture components fitted per site for the new domain

DEVICE = "cuda:0"
ARTIFACT_ID = "qwen3-0.6b-gmm1543k-int8"
MODEL_ID = "Qwen/Qwen3-0.6B"

DEMO = Path("lfa_demo").resolve()      # everything this notebook creates lives here
DATA, ARTIFACTS, WS = DEMO / "data", DEMO / "artifacts", DEMO / "workspace"
DEMO.mkdir(exist_ok=True)
print(DEMO)

/tmp/lfa-nb-run/cwd/lfa_demo


In [2]:
import logging, os, sys, time, textwrap

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logging.getLogger("datasets").setLevel(logging.WARNING)

import torch
import lfa

NOTEBOOK_STARTED = time.time()
print(f"lfa          {lfa.__version__}")
print(f"torch        {torch.__version__}")
print(f"CUDA device  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "this notebook needs a CUDA card"

/tmp/lfa-nb-run/pkg/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


lfa          0.1.0
torch        2.10.0+cu128
CUDA device  NVIDIA GeForce RTX 3090


## 1. The artifact: what makes adaptation data-free

LFA's preservation signal does not come from text. It comes from a file, built **once per
model**, that records what the hidden states arriving at each anchored sub-module look like:
a mean, a PCA basis with its eigenvalues, and a small mixture fitted in that basis, per site.
That file is the artifact, and the training loop samples hidden states from it instead of
reading a corpus.

`fetch_artifact` downloads the published one (~108 MB) and checks its SHA-256 against the
registry before putting it in place, because a truncated or wrong artifact has no visible
failure mode — the shapes would still be right, the run would still train, and only the
preservation number would be off.

A file already at the destination is **verified rather than re-downloaded**, so if you already
have a copy, drop it in as `lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt` (or point the
`LFA_ARTIFACT` environment variable at it) and the same call checksums it in place. The run
whose outputs are saved in this notebook took that route — hence the `copied …` line below and
the 0-second fetch — so what you are reading is the verify-in-place branch, not the download.
It is the same function and the same checksum either way.

In [3]:
import shutil
from lfa.artifact.fetch import ARTIFACTS as REGISTRY, fetch_artifact, list_artifacts

for entry in list_artifacts():
    print(f"{entry['id']:<28} {entry['size_mb']:>4} MB   {entry['kind']}")

# An out-of-band copy is accepted at the destination name; fetch_artifact then verifies it
# against the registry checksum instead of downloading it again.
local_copy = os.environ.get("LFA_ARTIFACT")
target = ARTIFACTS / f"{ARTIFACT_ID}.pt"
if local_copy and not target.exists():
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(local_copy, target)
    print(f"\ncopied {local_copy} -> {target}; it will be checksum-verified, not trusted")

t0 = time.time()
artifact_path = fetch_artifact(ARTIFACT_ID, ARTIFACTS)
print(f"\n{artifact_path}  ({artifact_path.stat().st_size / 1e6:.1f} MB, {time.time()-t0:.0f}s)")

qwen3-0.6b-diagonal             1 MB   diagonal (budget floor)
qwen3-0.6b-gmm1543k-int8      108 MB   correlated-linear + GMM K=32 MLP (int8)

copied /tmp/lfa-nb-run/artifact/qwen3-0.6b-gmm1543k-int8.pt -> /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt; it will be checksum-verified, not trusted
Artifact qwen3-0.6b-gmm1543k-int8 already present at /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt (checksum verified)



/tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt  (112.8 MB, 0s)


Worth opening, because the claim that this is "not the old data in disguise" is checkable.
The artifact is a dictionary of per-site blocks and one metadata block, and the inventory of
what a block may hold is printed at the bottom of the next cell: tensors of statistics, integer
counts, floats, and one string per site naming the kind of mixture that was fitted. No token
ids, no documents, nothing sequence-shaped. The object is a *distribution over vectors*, and
the sites it covers are the inputs of the sub-modules being preserved.

In [4]:
from lfa.artifact.schema import load_artifact

params = load_artifact(artifact_path)      # int8 on disk, dequantized on load
meta = params["__meta__"]
sites = [k for k in params if k != "__meta__"]

print("__meta__:")
for key, value in meta.items():
    print(f"   {key:<18} {value if not isinstance(value, list) else value}")
print(f"\n{len(sites)} site blocks, e.g. {sites[:3]} ... {sites[-2:]}")

example = params["12_pre_mlp"]
print("\nwhat one site stores (layer 12, the input of the whole MLP):")
for key, value in example.items():
    shape = tuple(value.shape) if torch.is_tensor(value) else value
    print(f"   {key:<22} {shape}")

types = {type(v).__name__ for block in sites for v in params[block].values()}
print(f"\nvalue types across every site block: {sorted(types)}")

__meta__:
   model_id           Qwen/Qwen3-0.6B
   hidden_size        1024
   num_layers         28
   sites              ['pre_qkv', 'pre_o', 'pre_mlp', 'pre_lm_head']
   n_samples_total    1543040
   built_with         research code; meta block added by lfa-anchoring
   lfa_version        0.1.0

84 site blocks, e.g. ['0_pre_o', '0_pre_mlp', '1_pre_qkv'] ... ['27_pre_mlp', '28_pre_lm_head']

what one site stores (layer 12, the input of the whole MLP):
   mean                   (1024,)
   std                    (1024,)
   pca_components         (1024, 747)
   pca_eigenvalues        (747,)
   pca_n_components       747
   gmm_n_components       32
   gmm_weights            (32,)
   gmm_means              (32, 747)
   gmm_covariances        (32, 747)
   gmm_covariance_type    diag
   gmm_log_likelihood_p10 -799.129150390625
   gmm_log_likelihood_p50 -675.5284423828125
   gmm_log_likelihood_p90 -548.2757568359375
   gaussian_coverage_ratio 1.2710726009969582
   gaussian_nn_distance_mean 2

## 2. Two domains, from Project Gutenberg

Two books, both public domain, both a single plain-text download with no account and no
credentials. They are chosen to be *far apart* — different century, different purpose,
different vocabulary — so that "adapted to A" and "adapted to B" are visibly different states
of the same model, and so that neither one is WikiText-2, which is the general axis.

| | domain A | domain B |
|---|---|---|
| book | *On the Origin of Species* | *Domestic Cookery, Useful Receipts, and Hints to Young Housekeepers* |
| author, year | Charles Darwin, 1859 | Elizabeth E. Lea, 1853 |
| Gutenberg id | 1228 | 9101 |
| download | `https://www.gutenberg.org/cache/epub/1228/pg1228.txt` (0.97 MB) | `https://www.gutenberg.org/cache/epub/9101/pg9101.txt` (0.48 MB) |

**1.45 MB of text in total.** The cell below downloads them, strips the Gutenberg licence
header and footer, and accumulates paragraphs into documents of about 3,500 characters —
roughly 850 tokens, so most documents are one or two of the 512-token chunks the trainer reads.

Every eighth document goes to a `heldout/` directory and is **never trained on, at either
stage**. Every domain perplexity printed in this notebook is measured on those documents, so
all of them are comparable with each other. (The trainer separately keeps a tenth of the
*training* documents back for its own per-epoch curve; that is a different split, used only to
choose the number of epochs.)

In [5]:
import hashlib, re, urllib.request

BOOKS = {
    "darwin":  dict(gid=1228, title="On the Origin of Species (Charles Darwin, 1859)"),
    "cookery": dict(gid=9101, title="Domestic Cookery (Elizabeth E. Lea, 1853)"),
}
_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
_END = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)


def strip_gutenberg(raw: str) -> str:
    """Drop the licence header and footer, keeping the book."""
    if (m := _START.search(raw)):
        raw = raw[m.end():]
    if (m := _END.search(raw)):
        raw = raw[:m.start()]
    return raw.strip()


def to_documents(text: str, target_chars: int = 3500, min_chars: int = 1200) -> list[str]:
    """Accumulate blank-line-separated paragraphs into documents of about `target_chars`."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    documents, buffer, size = [], [], 0
    for paragraph in paragraphs:
        buffer.append(paragraph)
        size += len(paragraph) + 2
        if size >= target_chars:
            documents.append("\n\n".join(buffer))
            buffer, size = [], 0
    if size >= min_chars:
        documents.append("\n\n".join(buffer))
    return [d for d in documents if len(d) >= min_chars]


def prepare(name: str, hold_out_every: int = 8) -> dict:
    spec = BOOKS[name]
    raw_path = DATA / "raw" / f"pg{spec['gid']}.txt"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not raw_path.exists():
        url = f"https://www.gutenberg.org/cache/epub/{spec['gid']}/pg{spec['gid']}.txt"
        with urllib.request.urlopen(url, timeout=120) as response:
            raw_path.write_bytes(response.read())
    raw = raw_path.read_bytes()
    documents = to_documents(strip_gutenberg(raw.decode("utf-8", errors="replace")))

    train, heldout = DATA / name / "train", DATA / name / "heldout"
    for directory in (train, heldout):
        directory.mkdir(parents=True, exist_ok=True)
        for stale in directory.glob("*.txt"):
            stale.unlink()
    for i, document in enumerate(documents):
        destination = heldout if i % hold_out_every == hold_out_every - 1 else train
        (destination / f"{i:04d}.txt").write_text(document, encoding="utf-8")

    return dict(title=spec["title"], mb=len(raw) / 1e6,
                sha256=hashlib.sha256(raw).hexdigest()[:12],
                train=len(list(train.glob("*.txt"))), heldout=len(list(heldout.glob("*.txt"))))


corpora = {name: prepare(name) for name in BOOKS}
for name, info in corpora.items():
    print(f"{name:<9} {info['title']}\n"
          f"          {info['mb']:.2f} MB downloaded, sha256 {info['sha256']}…, "
          f"{info['train']} training documents + {info['heldout']} held out")

DARWIN_TRAIN, DARWIN_HELD = DATA / "darwin" / "train", DATA / "darwin" / "heldout"
COOKERY_TRAIN, COOKERY_HELD = DATA / "cookery" / "train", DATA / "cookery" / "heldout"

darwin    On the Origin of Species (Charles Darwin, 1859)
          0.97 MB downloaded, sha256 ededa9c0bf87…, 193 training documents + 27 held out
cookery   Domestic Cookery (Elizabeth E. Lea, 1853)
          0.48 MB downloaded, sha256 8ab8a0d20e54…, 105 training documents + 15 held out


A paragraph of each, so it is clear how different the two domains are:

In [6]:
for name, path in (("A / darwin", DARWIN_TRAIN), ("B / cookery", COOKERY_TRAIN)):
    sample = sorted(path.glob("*.txt"))[6].read_text(encoding="utf-8")
    print(f"--- {name} " + "-" * (66 - len(name)))
    print(textwrap.fill(" ".join(sample.split())[:520], 88), "…\n")

--- A / darwin --------------------------------------------------------
Sterility has been said to be the bane of horticulture; but on this view we owe
variability to the same cause which produces sterility; and variability is the source of
all the choicest productions of the garden. I may add, that as some organisms will breed
most freely under the most unnatural conditions (for instance, the rabbit and ferret
kept in hutches), showing that their reproductive system has not been thus affected; so
will some animals and plants withstand domestication or cultivation, and vary very
slight …

--- B / cookery -------------------------------------------------------
Celery Sauce. Take a large bunch of celery, cut it fine, and boil it till soft, in a
pint of water; thicken it with butter and flour, and season it with salt, pepper, and
mace. Bacon Dumplings. Cut slices of cooked bacon, and pepper them; roll out crust as
for apple dumplings; slice some potatoes very thin, and put them in the cru

## 3. A workspace

A workspace is the state machine that keeps a chain honest. It records **which model the next
stage adapts**, **which version of `p(h)` that stage anchors against**, and **the order the two
may be done in** — get any of those wrong and the run still completes, still reports a
plausible loss, and quietly measures something else.

`init` does not load the model; it records it. It copies the artifact in as `artifacts/v1.pt`,
so the workspace owns its own `p(h)` and later versions sit beside it. It picks up the bundled
recipe whose `model_id` is this model automatically.

The CLI equivalent of this cell is one line:

```bash
lfa init lfa_demo/workspace --model Qwen/Qwen3-0.6B --artifact qwen3-0.6b-gmm1543k-int8
```

In [7]:
from lfa import Recipe, Workspace

workspace = Workspace.init(WS, MODEL_ID, artifact=str(artifact_path), artifact_id=ARTIFACT_ID)
recipe = Recipe.load("qwen3-0.6b")

print(f"model            {workspace.state['model_id']}")
print(f"artifact         v{workspace.state['artifact_version']}  ({workspace.state['artifact_id']})")
print(f"stage            {workspace.state['stage']}")
print("\nthe operating point this recipe carries:")
for field in ("lora_rank", "lora_alpha", "lambda_qkv", "lambda_mlp", "mu", "anchor_end_ratio",
              "n_anchor_samples", "epochs", "learning_rate", "batch_size", "sequence_length",
              "val_fraction", "stage2_lambda_multiplier"):
    print(f"   {field:<26} {getattr(recipe, field)}")

notes = recipe.warnings(rank=recipe.lora_rank, artifact_id=ARTIFACT_ID)
print(f"\noff-calibration warnings: {notes if notes else 'none — this is the point λ was tuned at'}")

Local artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt recorded as the published 'qwen3-0.6b-gmm1543k-int8'


Artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace over Qwen/Qwen3-0.6B


model            Qwen/Qwen3-0.6B
artifact         v1  (qwen3-0.6b-gmm1543k-int8)
stage            0

the operating point this recipe carries:
   lora_rank                  32
   lora_alpha                 64
   lambda_qkv                 100000.0
   lambda_mlp                 100000.0
   mu                         0.05
   anchor_end_ratio           0.1
   n_anchor_samples           16
   epochs                     15
   learning_rate              0.0003
   batch_size                 6
   sequence_length            512
   val_fraction               0.1
   stage2_lambda_multiplier   3.0

off-calibration warnings: none — this is the point λ was tuned at


## 4. Stage 1 — adapt to domain A

One training run. Rank-32 LoRA over the frozen teacher, with three terms in the loss:

```
L  =  L_content                                    cross-entropy on Darwin
   +  λ · E_{h ~ p(h)} ‖f_student(h) − f_teacher(h)‖²    the function anchor
   +  μ · ‖W_student − W_teacher‖²                 the weight backstop
```

The middle term never touches the batch. On every step it draws 16 hidden states per site
**from the artifact**, runs the teacher's sub-module and the student's sub-module on them, and
penalizes the difference. That is the whole of the preservation signal, and it is why nothing
in this notebook ever needs the old data.

### On the number of epochs

The recipe's 15 epochs were tuned on a corpus of about 6.6 MB. These two are 1.45 MB together,
so roughly a quarter of the text, and on that much they over-train.

Count documents rather than bytes and you get a much larger-looking gap, because that corpus is
1,673 files of which 1,596 are short generated question-answer pairs making up only 14 % of its
text; the substance is 77 long documents. A file count would say the recipe saw nine times more
than this notebook. It saw about four and a half times more text. Bytes are the honest measure
here, and even they overstate it slightly: those short files were dropped from most epochs by
the loader that trained that corpus, so the recipe saw rather less of them than their share of
the bytes suggests.

An 8-epoch pass
of exactly this configuration was run first, and the trainer's held-out perplexity bottomed at
epoch 4 (18.46, 17.64, 17.08, **16.84**) and then climbed — 16.93, 17.64, 17.94, 18.25.
`final_model` is the last epoch by design, so the fix is to re-run at the epoch it bottomed at,
which is what `EPOCHS = 4` above is. That choice was made on the **domain** axis of the
anchored run alone, before any control had been trained to compare it against, and the control
below is trained at the same 4 epochs. Section 8b returns to whether that is fair to it.

**That number moved when the loader was fixed**, and the move is the fix showing up: the same
pass under the old chunker bottomed at epoch 5 with 17.20 (18.51, 17.62, 17.60, 17.31, 17.20,
17.87, 18.99, 18.46). Training on all of every document instead of about 83 % of it reaches a
better minimum — 16.84 — and reaches it an epoch sooner, which is what more text per epoch
should do. (A 4-epoch run is not the first four epochs of that pass: the learning-rate schedule is
laid over whatever number of epochs you ask for. Its own curve, printed in section 8b, bottoms at
epoch 3 and ends 0.8 % above that — inside the 10 % the trainer treats as a wobble rather than a
turn, and it prints nothing.)

The CLI equivalent:

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/train --epochs 4
```

In [8]:
t0 = time.time()
stage1 = workspace.train(DARWIN_TRAIN, epochs=EPOCHS, device=DEVICE)
stage1_seconds = time.time() - t0

print(f"\nstage {stage1['stage']}: {stage1['n_train_docs']} documents trained, "
      f"{stage1['n_val_docs']} held back for the curve, λ = {stage1['lambda_applied']:,.0f}, "
      f"{stage1_seconds/60:.1f} min")

loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


loader: a document that fits in one chunk is trained whole in every epoch; under keep_short_whole=False it is cut at the epoch's chunk offset like a longer one


Stage 1: Qwen/Qwen3-0.6B on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/train (artifact v1, λ_qkv=100000.0, 4 epochs)


Layer weights (cosine, end_ratio=0.1): L0=0.0631, L27=0.0065


LR schedule: warmup(50) → cosine(210)


...

  Held-out: loss=2.8364 perplexity=17.054 over 18042 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/final_model


  Training state saved (epoch=4, step=310, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/training_history.json



stage 1: 173 documents trained, 20 held back for the curve, λ = 100,000, 4.1 min


## 5. The control that makes the point

`evaluate` reads the stage on **both axes at once**, because either alone is meaningless: a run
that reports only the domain has not said what it gave up, and one that reports only WikiText-2
has not said whether it learned anything.

`compare_unanchored=True` adds a third column by training the stage **again** with
λ = μ = 0 — same corpus, same 4 epochs, same learning rate, same seed, same everything except
the anchor. That is a second full training run, and it is what says what the anchor bought.
Same corpus, same steps, so the difference is the anchor — with one caveat that section 8b
measures rather than assumes: 4 epochs is the *anchored* run's best dose, not the control's.

The domain axis here is Darwin's `heldout/` directory, passed explicitly. (`evaluate` logs
"a fit, not a held-out measurement" whenever a corpus is named, because a named corpus is not
split — but these documents were never in the training directory at all, so the number is
genuinely held out. Naming it is what makes it the *same* text as every later domain-A read.)

```bash
lfa evaluate --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/heldout \
             --compare-unanchored --n-windows 200
```

In [9]:
t0 = time.time()
scores1 = workspace.evaluate(corpus=DARWIN_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control1_seconds = time.time() - t0
print("\n" + scores1["table"])

Evaluating stage 1 on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/heldout (64 chunks, trained on -- a fit, not a held-out measurement)


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


WikiText-2 perplexity 24.92 (loss 3.2157 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 17.10 (loss 2.8391 over 26147 tokens in 64 chunks)



| metric               | before | after |     Δ% | unanchored |     Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | -----: |
| general (WikiText-2) |  17.80 | 16.09 |  -9.6% |      24.92 | +40.0% |
| domain               |  30.12 | 17.45 | -42.1% |      17.10 | -43.2% |


In [10]:
before, after, unanchored = scores1["before"], scores1["after"], scores1["unanchored"]
print(f"""
Read it as two questions.

  Did it learn Darwin?   {before['domain']:.2f} -> {after['domain']:.2f}
                         ({(after['domain']/before['domain'] - 1) * 100:+.1f}% held-out domain perplexity)

  What did that cost?    WikiText-2 {before['general']:.2f} -> {after['general']:.2f}
                         ({(after['general']/before['general'] - 1) * 100:+.1f}%)

  Without the anchor:    Darwin      {before['domain']:.2f} -> {unanchored['domain']:.2f}   ({(unanchored['domain']/before['domain'] - 1) * 100:+.1f}%)
                         WikiText-2  {before['general']:.2f} -> {unanchored['general']:.2f}   ({(unanchored['general']/before['general'] - 1) * 100:+.1f}%)
""")


Read it as two questions.

  Did it learn Darwin?   30.12 -> 17.45
                         (-42.1% held-out domain perplexity)

  What did that cost?    WikiText-2 17.80 -> 16.09
                         (-9.6%)

  Without the anchor:    Darwin      30.12 -> 17.10   (-43.2%)
                         WikiText-2  17.80 -> 24.92   (+40.0%)



## 6. `extend` — folding A in, so that B never has to see it

A second domain is not a second training run. Stage 2 has to adapt the model that already
carries A, and it has to anchor against a `p(h)` that *describes* that model — otherwise it
preserves a function the model no longer computes. `extend` does both, and the workspace
refuses to train a different corpus until it has:

* **Merge.** Stage 1's adapter is merged into the model it trained over, giving
  `models/stage1_fused` — a plain checkpoint. That is what stage 2 adapts.
* **Extend `p(h)`.** Domain A is run through the *fused* model, its activations are fitted as a
  small mixture **in the base artifact's own PCA basis**, and the two mixtures are merged as a
  sample-weighted union:

  ```
  p(h) = (1 − a) · p_base(h) + a · p_domain(h),      a = n_domain / (n_base + n_domain)
  ```

The union is exact — it is the distribution of "draw from the base pool with probability
1 − a, else from the domain pool" — so nothing is refitted, and the merged artifact is a
sufficient statistic for the round after that as well. Each round reads only the **new** domain
and adds a fixed number of components per site, so the cost per round is flat in the number of
rounds. Replay has to rehearse every prior corpus at every stage; this does not.

The output is `artifacts/v2.pt`, and it is still a set of per-site tensors. **No Darwin text
is stored anywhere in the workspace from here on** — the cell after this one checks that.

`--need` is the memory bill: the collection keeps one `need x width` float32 buffer
per site, on the host, for all 84 sites at once — about 9 GB of RAM at the 20,000
used here, and about 18 GB at the 40,000 default.

```bash
lfa extend --workspace lfa_demo/workspace --need 20000 --k-domain 4
```

In [11]:
t0 = time.time()
artifact_v2 = workspace.extend(need=EXTEND_NEED, k_domain=EXTEND_K, device=DEVICE)
extend_seconds = time.time() - t0
print(f"\n{artifact_v2}  ({artifact_v2.stat().st_size / 1e6:.1f} MB, {extend_seconds/60:.1f} min)")
print(f"current model    {workspace.state['current_model']}")
print(f"current artifact v{workspace.state['artifact_version']}")

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/final_model into Qwen/Qwen3-0.6B


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Collected 20000 activations at each of 84 sites from 52 chunks


Fitting the domain mixture: site 1/84 (0_pre_o)


...

Fitting the domain mixture: site 80/84 (26_pre_mlp)


Fitting the domain mixture: site 84/84 (28_pre_lm_head)


Extended 84 sites with K 4-4 new components from 20000 activations (domain weight share 0.0128); wrote version 2


Stage 1 folded in: model /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused, artifact v2



/tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v2.pt  (113.3 MB, 1.1 min)
current model    /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused
current artifact v2


In [12]:
v1, v2 = load_artifact(artifact_path), load_artifact(artifact_v2)
site = "12_pre_mlp"
basis_shift = (v1[site]["pca_components"].float()
               - v2[site]["pca_components"].float()).abs().max().item()

print(f"{site}: mixture components  v1 {v1[site]['gmm_n_components']}"
      f"  ->  v2 {v2[site]['gmm_n_components']}   (+{EXTEND_K} for domain A)")
print(f"          basis: largest entry-wise change {basis_shift:.2e} — the domain mixture is")
print(f"                 fitted IN the base basis, so this is int8 round-trip noise, not a refit")
print(f"          value types: {sorted({type(x).__name__ for x in v2[site].values()})}")


# The whole point of the method, stated as a filesystem fact: search every byte the workspace
# holds — adapters, optimizer state, the fused model, both artifacts — for a phrase the domain-A
# corpus is full of.
def contains(path, needle, chunk=1 << 22):
    with path.open("rb") as handle:
        tail = b""
        while (block := handle.read(chunk)):
            if needle in tail + block:
                return True
            tail = block[-len(needle):]
    return False


NEEDLE = b"natural selection"
files = [p for p in WS.rglob("*") if p.is_file()]
carriers = [p.relative_to(WS) for p in files if contains(p, NEEDLE)]
occurrences = sum(doc.read_bytes().count(NEEDLE) for doc in DARWIN_TRAIN.glob("*.txt"))

print(f"\n{NEEDLE.decode()!r} appears {occurrences} times in the domain-A training corpus.")
print(f"Files in the workspace ({len(files)} files, "
      f"{sum(p.stat().st_size for p in files) / 1e9:.2f} GB) that contain it: {len(carriers)}")

12_pre_mlp: mixture components  v1 32  ->  v2 36   (+4 for domain A)
          basis: largest entry-wise change 0.00e+00 — the domain mixture is
                 fitted IN the base basis, so this is int8 round-trip noise, not a refit
          value types: ['Tensor', 'float', 'int', 'str']



'natural selection' appears 186 times in the domain-A training corpus.
Files in the workspace (26 files, 1.92 GB) that contain it: 0


## 7. Stage 2 — adapt to domain B

Same command as stage 1. Three things are different, and the workspace supplies all three:

* it adapts `models/stage1_fused`, not the base model;
* it anchors against `artifacts/v2.pt`, which now describes *that* model on *both* domains;
* λ is multiplied by the recipe's `stage2_lambda_multiplier` (3.0), because a later stage is
  holding more than the base model's functions in place. That multiplier is a **level, not a
  compounding factor** — stage 3 would anchor at the same multiple, not its square — and the
  shipped 3.0 is a starting default rather than a calibrated constant.

Nothing from Darwin is read. There is no buffer, no rehearsal set, no replay mixture.

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/cookery/train --epochs 4
```

In [13]:
t0 = time.time()
stage2 = workspace.train(COOKERY_TRAIN, epochs=EPOCHS, device=DEVICE)
stage2_seconds = time.time() - t0

print(f"\nstage {stage2['stage']}: {stage2['n_train_docs']} documents, "
      f"λ = {stage2['lambda_applied']:,.0f} "
      f"(= {recipe.lambda_qkv:,.0f} × {recipe.stage2_lambda_multiplier}), "
      f"anchored against artifact v{stage2['artifact_version']}, {stage2_seconds/60:.1f} min")
print(f"          starting model: {stage2['base_model']}")

loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


loader: a document that fits in one chunk is trained whole in every epoch; under keep_short_whole=False it is cut at the epoch's chunk offset like a longer one


Stage 2: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/train (artifact v2, λ_qkv=300000.0, 4 epochs)


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Layer weights (cosine, end_ratio=0.1): L0=0.0631, L27=0.0065


...

  Held-out: loss=2.6623 perplexity=14.330 over 10503 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/final_model


  Training state saved (epoch=4, step=170, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/training_history.json



stage 2: 94 documents, λ = 300,000 (= 100,000 × 3.0), anchored against artifact v2, 2.3 min
          starting model: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


## 8. The contrast again, and the question the whole notebook is about

First the same two-axis read as before, with its own unanchored control: stage 2 re-run from
the same starting model with λ = μ = 0.

In [14]:
t0 = time.time()
scores2 = workspace.evaluate(corpus=COOKERY_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control2_seconds = time.time() - t0
print("\n" + scores2["table"])

The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Evaluating stage 2 on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/heldout (33 chunks, trained on -- a fit, not a held-out measurement)


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


WikiText-2 perplexity 22.59 (loss 3.1174 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 13.37 (loss 2.5927 over 14949 tokens in 33 chunks)



| metric               | before | after |     Δ% | unanchored |     Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | -----: |
| general (WikiText-2) |  16.09 | 15.94 |  -0.9% |      22.59 | +40.4% |
| domain               |  23.03 | 15.13 | -34.3% |      13.37 | -42.0% |


That table's "before" column is the model **after domain A**, so it reads stage 2 in isolation.
It does not yet answer the question the method exists for, which is about A.

So: five models, three axes, one table. The general axis is WikiText-2, which no stage ever
trained on. The two domain axes are the `heldout/` documents, which no stage ever trained on
either. Every row is the same three measurements on the same three texts.

In [15]:
import gc
from peft import PeftModel
from lfa.corpus import load_corpus
from lfa.evaluate import domain_perplexity, wikitext2_perplexity
from lfa.models import load_teacher, load_tokenizer

tokenizer = load_tokenizer(MODEL_ID)


def held_out(path):
    corpus, _ = load_corpus(path, tokenizer, max_length=recipe.sequence_length,
                            val_fraction=0.0, seed=recipe.seed,
                            keep_short_whole=recipe.keep_short_whole)
    return corpus


DOMAINS = {"A · Darwin": held_out(DARWIN_HELD), "B · cookery": held_out(COOKERY_HELD)}
STAGE1_FUSED = WS / "models" / "stage1_fused"

MODELS = {
    "base Qwen3-0.6B":            (MODEL_ID, None),
    "after A — anchored":         (MODEL_ID, WS / "runs" / "stage1" / "final_model"),
    "after A — NO anchor":        (MODEL_ID, WS / "runs" / "stage1_unanchored" / "final_model"),
    "after A+B — anchored":       (STAGE1_FUSED, WS / "runs" / "stage2" / "final_model"),
    "after A+B — NO anchor on B": (STAGE1_FUSED, WS / "runs" / "stage2_unanchored" / "final_model"),
}


def score(base, adapter):
    model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    try:
        row = {"WikiText-2": wikitext2_perplexity(model, tokenizer, n_windows=N_WINDOWS,
                                                  device=DEVICE)}
        row.update({name: domain_perplexity(model, tokenizer, corpus, device=DEVICE)
                    for name, corpus in DOMAINS.items()})
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


logging.getLogger("lfa.evaluate").setLevel(logging.WARNING)   # the table is the output here
t0 = time.time()
table = {label: score(*spec) for label, spec in MODELS.items()}
final_table_seconds = time.time() - t0

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


In [16]:
columns = ["WikiText-2", "A · Darwin", "B · cookery"]
width = max(len(label) for label in table)
base_row = table["base Qwen3-0.6B"]

header = f"| {'model':<{width}} | " + " | ".join(f"{c:>18}" for c in columns) + " |"
print(header)
print(f"| {'-' * width} | " + " | ".join("-" * 18 for _ in columns) + " |")
for label, row in table.items():
    cells = []
    for column in columns:
        delta = (row[column] / base_row[column] - 1) * 100
        cells.append(f"{row[column]:8.2f}" + ("         " if label.startswith("base")
                                              else f" ({delta:+5.1f}%)"))
    print(f"| {label:<{width}} | " + " | ".join(f"{c:>18}" for c in cells) + " |")
print("\nperplexity, lower is better; % is against the base model on the same text")

| model                      |         WikiText-2 |         A · Darwin |        B · cookery |
| -------------------------- | ------------------ | ------------------ | ------------------ |
| base Qwen3-0.6B            |     17.80          |     30.12          |     26.51          |
| after A — anchored         |     16.09 ( -9.6%) |     17.45 (-42.1%) |     23.02 (-13.2%) |
| after A — NO anchor        |     24.92 (+40.0%) |     17.10 (-43.2%) |     32.51 (+22.6%) |
| after A+B — anchored       |     15.94 (-10.5%) |     18.92 (-37.2%) |     15.13 (-42.9%) |
| after A+B — NO anchor on B |     22.59 (+26.9%) |     31.45 ( +4.4%) |     13.37 (-49.6%) |

perplexity, lower is better; % is against the base model on the same text


### How to read that table

Four comparisons, in the order they answer the notebook's question.

1. **Row 2 against row 1** — did the anchored stage learn Darwin, and what did WikiText-2 do?
2. **Row 3 against row 2** — the control. Same corpus, same steps, anchor off. The unanchored
   run reaches the domain *harder*; the general column is where it pays for it. Read that gap
   as an **upper bound** on what the anchor bought, not as the anchor alone: both controls are
   trained at the anchored run's number of epochs, which is past their own optimum. Section 8b
   re-runs them at their own dose and separates the two.
3. **Row 4's `A · Darwin` cell against row 2's** — the chain question. Stage 2 trained on
   nothing but recipes, and read no Darwin text at all. How much of A survived?
4. **Row 5's `A · Darwin` cell against row 4's** — the same stage without the anchor. Both start
   from the same model that already knows A; only one of them is holding it. The same upper-bound
   caveat applies, and section 8b measures this one too.

A caution that belongs beside the numbers: **this is one run per cell at one seed**, on corpora
roughly a quarter the size of the corpus the operating point was tuned on, at a quarter of
its epochs.
Read the *direction and size* of each contrast, not the digits. Two more warnings from
`docs/concepts.md` apply directly here: a general perplexity *below* the base model's is not by
itself evidence that anything was preserved, and over-anchoring makes the general axis look its
best while the domain quietly collapses — which is why nothing in this notebook is read on one
axis alone.

In [17]:
a_after_a = table["after A — anchored"]["A · Darwin"]
a_after_ab = table["after A+B — anchored"]["A · Darwin"]
a_after_ab_free = table["after A+B — NO anchor on B"]["A · Darwin"]
g_anchored = table["after A — anchored"]["WikiText-2"]
g_free = table["after A — NO anchor"]["WikiText-2"]

print(f"""
1. stage 1, anchored     Darwin {base_row['A · Darwin']:.2f} -> {a_after_a:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_anchored:.2f}
2. stage 1, no anchor    Darwin {base_row['A · Darwin']:.2f} -> {table['after A — NO anchor']['A · Darwin']:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_free:.2f}
   what the anchor bought on the general axis: {(g_free / g_anchored - 1) * 100:+.1f}% worse without it

3. domain A across stage 2, anchored:     {a_after_a:.2f} -> {a_after_ab:.2f}   ({(a_after_ab / a_after_a - 1) * 100:+.1f}%)
4. domain A across stage 2, anchor off:   {a_after_a:.2f} -> {a_after_ab_free:.2f}   ({(a_after_ab_free / a_after_a - 1) * 100:+.1f}%)
   stage 2 read 0 words of Darwin in either case.
""")


1. stage 1, anchored     Darwin 30.12 -> 17.45    WikiText-2 17.80 -> 16.09
2. stage 1, no anchor    Darwin 30.12 -> 17.10    WikiText-2 17.80 -> 24.92
   what the anchor bought on the general axis: +54.9% worse without it

3. domain A across stage 2, anchored:     17.45 -> 18.92   (+8.4%)
4. domain A across stage 2, anchor off:   17.45 -> 31.45   (+80.3%)
   stage 2 read 0 words of Darwin in either case.



## 8b. Is the control fair? Both stages, at their own dose

There is a real objection to everything above, and it is worth measuring rather than waving at.

`EPOCHS = 4` was chosen from the **anchored** stage-1 run's held-out curve. Both unanchored
controls are trained at that same 4, because "the same run without the anchor" is what a control
is — but 4 is not *their* best dose. Without the anchor the model reaches the domain much faster
and then over-fits, and the trainer said so itself at the end of the stage-1 control ("this run
trained past its own optimum", lowest at epoch 2 and 12 % above it by epoch 4). The stage-2
control ends only 5 % above its own minimum — inside the 10 % the trainer treats as a wobble, so
it says nothing there — but its curve turns at epoch 3 all the same, and that is the dose the
cell below gives it. So part of every gap above is the anchor and part of it is three
extra epochs of unregularized training. Both curves are printed below, so the size of that
effect is visible rather than asserted.

The honest version of the question is: **at each run's own best dose, what is left?** Two more
runs answer it — each control again at the epoch *its own* curve bottomed at, read off that curve
in the cell below rather than written in, with its own complete learning-rate schedule, and each
in a workspace of its own so the chain built above is untouched.

In [18]:
import dataclasses
import json as _json


def curve(run_dir):
    return [(e["epoch"], e["val_perplexity"])
            for e in _json.loads((Path(run_dir) / "training_history.json").read_text())
            if "val_perplexity" in e]


print("held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set")
for stage, runs in (("stage 1 (darwin) ", ("stage1", "stage1_unanchored")),
                    ("stage 2 (cookery)", ("stage2", "stage2_unanchored"))):
    print(f"  {stage}")
    for label, run in zip(("anchored ", "no anchor"), runs):
        print(f"     {label}  " + "  ".join(f"e{n}:{v:6.2f}" for n, v in curve(WS / "runs" / run)))

held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set
  stage 1 (darwin) 
     anchored   e1: 18.52  e2: 17.38  e3: 16.91  e4: 17.05
     no anchor  e1: 16.57  e2: 14.80  e3: 15.53  e4: 16.62
  stage 2 (cookery)
     anchored   e1: 15.93  e2: 14.86  e3: 14.35  e4: 14.33
     no anchor  e1: 13.70  e2: 12.17  e3: 11.91  e4: 12.54


In [19]:
control = dataclasses.replace(recipe, lambda_qkv=0.0, lambda_mlp=0.0, mu=0.0)

# Each control's own best dose, read off ITS OWN curve above -- the rule that chose the anchored
# run's EPOCHS, applied to the run being asked about. Derived here rather than written in, so it
# cannot be inherited from an earlier run of this notebook.
DOSE_A = min(curve(WS / "runs" / "stage1_unanchored"), key=lambda point: point[1])[0]
DOSE_B = min(curve(WS / "runs" / "stage2_unanchored"), key=lambda point: point[1])[0]
print(f"each control's own best dose, from its own curve: stage 1 {DOSE_A} epochs, "
      f"stage 2 {DOSE_B} epochs (both anchored runs used {EPOCHS})")

t0 = time.time()

# Stage 1's control at its own dose: the base model, DOSE_A epochs, no anchor.
FAIR1 = DEMO / "workspace_dose_a"
fair1 = Workspace.init(FAIR1, MODEL_ID, artifact=str(artifact_path), artifact_id=ARTIFACT_ID)
fair1.train(DARWIN_TRAIN, recipe=control, epochs=DOSE_A, device=DEVICE)

# Stage 2's control at its own dose: the SAME starting model stage 2 had (the anchored stage-1
# fused checkpoint) and the same corpus, DOSE_B epochs, no anchor. A workspace of its own, so
# the chain above keeps pointing at its real stage 2.
FAIR2 = DEMO / "workspace_dose_b"
fair2 = Workspace.init(FAIR2, str(STAGE1_FUSED), artifact=str(artifact_v2),
                       artifact_id=ARTIFACT_ID, recipe="qwen3-0.6b")
fair2.train(COOKERY_TRAIN, recipe=control, epochs=DOSE_B, device=DEVICE)

dose_seconds = time.time() - t0
print(f"\n{dose_seconds / 60:.1f} min")
print(f"  stage-1 control, {DOSE_A} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR1 / "runs" / "stage1")))
print(f"  stage-2 control, {DOSE_B} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR2 / "runs" / "stage1")))

each control's own best dose, from its own curve: stage 1 2 epochs, stage 2 3 epochs (both anchored runs used 4)
Local artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt recorded as the published 'qwen3-0.6b-gmm1543k-int8'


Artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a over Qwen/Qwen3-0.6B


loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


...

  Held-out: loss=2.4974 perplexity=12.151 over 10503 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/final_model


  Training state saved (epoch=3, step=123, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/training_history.json



3.5 min
  stage-1 control, 2 epochs: e1: 16.59  e2: 15.31
  stage-2 control, 3 epochs: e1: 13.73  e2: 12.03  e3: 12.15


In [20]:
dose_a = score(MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model")
dose_b = score(STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model")


def block(title, rows, columns):
    width = max(len(label) for label, _ in rows)
    print(f"\n{title}")
    print(f"| {'':<{width}} | " + " | ".join(f"{c:>12}" for c in columns) + " |")
    print(f"| {'-' * width} | " + " | ".join("-" * 12 for _ in columns) + " |")
    for label, row in rows:
        print(f"| {label:<{width}} | " + " | ".join(f"{row[c]:12.2f}" for c in columns) + " |")


block("stage 1 — adapting the BASE model to Darwin",
      [("base model", base_row),
       (f"anchored, {EPOCHS} epochs", table["after A — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A — NO anchor"]),
       (f"no anchor, {DOSE_A} epochs", dose_a)],
      ["WikiText-2", "A · Darwin"])

block("stage 2 — adapting the same after-A model to the cookbook",
      [("model it started from", table["after A — anchored"]),
       (f"anchored, {EPOCHS} epochs", table["after A+B — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A+B — NO anchor on B"]),
       (f"no anchor, {DOSE_B} epochs", dose_b)],
      ["WikiText-2", "A · Darwin", "B · cookery"])

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).



stage 1 — adapting the BASE model to Darwin
|                     |   WikiText-2 |   A · Darwin |
| ------------------- | ------------ | ------------ |
| base model          |        17.80 |        30.12 |
| anchored, 4 epochs  |        16.09 |        17.45 |
| no anchor, 4 epochs |        24.92 |        17.10 |
| no anchor, 2 epochs |        18.87 |        15.70 |

stage 2 — adapting the same after-A model to the cookbook
|                       |   WikiText-2 |   A · Darwin |  B · cookery |
| --------------------- | ------------ | ------------ | ------------ |
| model it started from |        16.09 |        17.45 |        23.02 |
| anchored, 4 epochs    |        15.94 |        18.92 |        15.13 |
| no anchor, 4 epochs   |        22.59 |        31.45 |        13.37 |
| no anchor, 3 epochs   |        19.75 |        27.02 |        12.99 |


Read those on **all** the columns at once, which is the whole discipline of the method. The
question is never "which run has the better WikiText-2"; it is "at comparable fit on the domain
being learned, which run kept more of everything else". A control that is worse everywhere has
been beaten outright. A control that is *better* on the new domain and worse elsewhere has
bought the domain by spending what it already had — which is the trade the anchor exists to
change, not a trade it claims to abolish.

Two things this section is honest about, whichever way the numbers fell:

* Some of the gap in section 5 and section 8 is dose, not anchor. Reporting only the 5-epoch
  controls would have overstated the anchor by the difference between the two control rows here.
* A 2-epoch run is not simply a truncated 5-epoch run: the learning-rate schedule is laid over
  whatever number of epochs you ask for, so these are complete runs with their own warmup and
  decay. That is the right comparison, and it is also a *second* difference from the 5-epoch
  control, not a clean single-variable change.

And, still: one seed, one run per row, 193 documents.

## 9. Export: a plain checkpoint

`fuse` merges the last stage's adapter into the model it trained over and writes an ordinary
Hugging Face checkpoint — no PEFT wrapper, no `lfa` at load time, nothing to configure. It
loads with `AutoModelForCausalLM.from_pretrained` like any other model.

```bash
lfa fuse --workspace lfa_demo/workspace
```

In [21]:
exported = workspace.fuse()
print(exported)
print(sorted(p.name for p in exported.iterdir()))

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/final_model into /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export


/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export
['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = AutoModelForCausalLM.from_pretrained(str(exported), dtype=torch.bfloat16).to(DEVICE)
merged_tokenizer = AutoTokenizer.from_pretrained(str(exported))

PROMPTS = [
    "The several difficulties which have been discussed, namely, that",   # domain A
    "To make a plain beef soup, take",                                    # domain B
]
for prompt in PROMPTS:
    ids = merged_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = merged.generate(**ids, max_new_tokens=60, do_sample=False,
                              pad_token_id=merged_tokenizer.eos_token_id)
    print("· " + textwrap.fill(merged_tokenizer.decode(out[0], skip_special_tokens=True),
                               88, subsequent_indent="  ") + "\n")

del merged
gc.collect()
torch.cuda.empty_cache()

The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


· The several difficulties which have been discussed, namely, that of the structure of the
  brain, of the nerves, of the cerebrum, of the spinal cord, of the nerves of the spinal
  cord, of the nerves of the brain, of the nerves of the spinal cord, of the nerves of
  the brain, of the



· To make a plain beef soup, take a pound of beef, cut it up, and put it in a pot with a
  pound of water, and a pound of flour, and a little salt, and a little pepper; boil it
  for half an hour, and when it is done, put in a pound of onions,



Those two continuations are **qualitative only** — a greedy decode from one small model, not a
measurement, and nothing in this package scores generated text. They are here because it is
worth seeing that one set of weights answers in both registers.

## 10. The whole chain, and what it cost

In [23]:
import json

for entry in json.load((WS / "history.json").open()):
    perplexity = entry.get("perplexity") or {}
    after = (perplexity.get("after") or {})
    print(f"stage {entry['stage']}  {Path(entry['corpus']).parent.name:<9} "
          f"λ={entry['lambda_applied']:>9,.0f}  artifact v{entry['artifact_version']}  "
          f"{entry['n_train_docs']:>3} docs  "
          f"domain {after.get('domain', float('nan')):.2f}  "
          f"general {after.get('general', float('nan')):.2f}")

print(f"""
wall clock
  stage 1 (anchored)                {stage1_seconds / 60:5.1f} min
  stage 1 evaluate + control        {control1_seconds / 60:5.1f} min
  extend (fuse + fit + merge p(h))  {extend_seconds / 60:5.1f} min
  stage 2 (anchored)                {stage2_seconds / 60:5.1f} min
  stage 2 evaluate + control        {control2_seconds / 60:5.1f} min
  the five-model table              {final_table_seconds / 60:5.1f} min
  the two dose-matched controls     {dose_seconds / 60:5.1f} min
  ------------------------------------------
  notebook, end to end              {(time.time() - NOTEBOOK_STARTED) / 60:5.1f} min
""")

stage 1  darwin    λ=  100,000  artifact v1  173 docs  domain 17.45  general 16.09
stage 2  cookery   λ=  300,000  artifact v2   94 docs  domain 15.13  general 15.94

wall clock
  stage 1 (anchored)                  4.1 min
  stage 1 evaluate + control          4.9 min
  extend (fuse + fit + merge p(h))    1.1 min
  stage 2 (anchored)                  2.3 min
  stage 2 evaluate + control          3.2 min
  the five-model table                2.4 min
  the two dose-matched controls       3.5 min
  ------------------------------------------
  notebook, end to end               22.6 min



## 11. What the three models actually say

Everything above is a perplexity. Perplexity is the right instrument for this method — it is
deterministic, it reads all three axes at once, and sections 5, 8 and 8b are argued on it. What
it does not tell you is what the models *sound* like, and that is worth seeing too.

So: five probes per stage, put to three models each under identical settings — **greedy decoding,
200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`**, the chat template with
Qwen3's thinking mode off. Two of the ten slots are deliberate repeats (`A1` asks about domain A
again after domain B has been learned; `G1` is the same general question in both panels), which
makes thirty generations from eight questions. The probe set was written down before any output
was looked at, and nothing in it was reworded, dropped or swapped afterwards. Every generation is
printed in full, including the ones that cut against the story.

**Why those two repetition controls.** Plain greedy decoding with no repetition penalty makes a
0.6B model repeat a clause until the token budget runs out, and it did: an earlier decode of these
same ten probes had every trained arm looping, one of them five times over. That is a property of
the decoder, not of any model, and it makes the panel unreadable. The penalty (the standard mild
band) and the 6-token no-repeat window are set **identically for every model**, including where
they cost one — they were chosen on whether they clear the loops, not on whose answers they
improve, and the base model pays for them too (see `G2`). Decoding stays deterministic, so there
is no seed to choose and no run to repeat until it looks good.

Three families, so it is clear what to look at:

| | |
|---|---|
| `A*`, `B*` | **adaptation** — does the model know the domain it was just trained on |
| `G*` | **retained general capability** — the axis forgetting destroys first |
| `I*` | **integration** — domain content delivered inside behaviour the base model already had: register control, brevity, doing what was asked |

**This is qualitative illustration, not evidence.** One greedy sample per model per prompt is not
a population, nothing here is scored by anything, and no claim in this notebook rests on it. The
measurements are the tables above.

One thing the second panel needs said plainly: **both** of its trained models start from the
*anchored* stage-1 checkpoint. It isolates stage-2 anchoring on its own; it is not an anchored
chain against an unanchored one.

In [24]:
# The probe set is fixed before any output is seen, and nothing below is dropped or shortened
# afterwards. `A*`/`B*` ask whether the domain was learned; `G*` ask whether general capability
# survived, which is the axis forgetting destroys first; `I*` ask for domain content delivered
# inside behaviour the base model already had -- register, brevity, following the instruction.
DARWIN_Q = "What did Darwin mean by natural selection?"
GENERAL_Q = "What is photosynthesis? Answer in two sentences."

PANEL_A = [
    ("A1", DARWIN_Q),
    ("A2", "According to The Origin of Species, what part does the struggle for existence play "
           "in the modification of species?"),
    ("G1", GENERAL_Q),
    ("G2", "List three prime numbers greater than 10."),
    ("I1", "In two sentences, explain to a ten-year-old why some animals in a species survive "
           "and others do not."),
]

PANEL_B = [
    ("A1", DARWIN_Q),                                    # the retention probe, repeated
    ("B1", "How do you make a plain suet pudding?"),
    ("B2", "What is the proper way to roast a joint of beef?"),
    ("G1", GENERAL_Q),                                   # repeated
    ("I2", "Give a short recipe for a simple soup, then explain in one sentence why boiling "
           "changes the food."),
]


# Decoding, identical for every model and every probe. Still greedy, so the section is
# deterministic and reproduces on a re-run; the two repetition controls are there because plain
# greedy decoding makes a 0.6B model repeat a clause until the budget runs out, which is a
# property of the decoder and would be easy to misread as a property of a model.
GENERATION = dict(
    max_new_tokens=200,        # unchanged -- room for a numbered recipe or a short explanation
    do_sample=False,           # greedy: no seed, no run-to-run variation, nothing to re-roll
    repetition_penalty=1.15,   # the standard mild band; 1.1 left some answers still looping
    no_repeat_ngram_size=6,    # hard block on repeating a 6-token span -- short domain phrases
                               # such as "struggle for existence" still come through
)


def answer(model, question):
    """One deterministic answer, chat template, thinking off -- GENERATION for every model."""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": question}],
                                           tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, pad_token_id=tokenizer.eos_token_id, **GENERATION)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_panel(models, probes):
    """Every model's answer to every probe. Loads one model at a time; nothing is filtered."""
    answers = {}
    for label, (base, adapter) in models.items():
        model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
        if adapter is not None:
            model = PeftModel.from_pretrained(model, str(adapter))
        try:
            answers[label] = {tag: answer(model, question) for tag, question in probes}
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    return answers


def show(answers, probes):
    for tag, question in probes:
        print(f"\n{'=' * 96}\n{tag}  {question}\n{'=' * 96}")
        for label, by_tag in answers.items():
            body = textwrap.fill(by_tag[tag], 92, initial_indent="    ", subsequent_indent="    ")
            print(f"\n  [{label}]\n{body}")


STAGE1_PANEL = {
    "base":       (MODEL_ID, None),
    "unanchored": (MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model"),
    "anchored":   (STAGE1_FUSED, None),
}

t0 = time.time()
answers_a = run_panel(STAGE1_PANEL, PANEL_A)
print(f"stage-1 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  base        Qwen3-0.6B, untouched\n"
      f"  unanchored  domain A with lambda = mu = 0, at its own best dose ({DOSE_A} epochs)\n"
      f"  anchored    domain A with the recipe's anchor ({EPOCHS} epochs)")
show(answers_a, PANEL_A)

stage-1 panel: 3 models x 5 probes, 54 s
  base        Qwen3-0.6B, untouched
  unanchored  domain A with lambda = mu = 0, at its own best dose (2 epochs)
  anchored    domain A with the recipe's anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [base]
    Charles Darwin's concept of **natural selection** is a fundamental principle in
    evolutionary biology, and it has had a profound impact on the understanding of life on
    Earth. Here’s what he meant:  ### 1. **The Origin of Species**:      Darwin proposed
    that species evolve over time through **natural selection**, where organisms with traits
    better suited to their environment are more likely to survive and reproduce. This leads
    to the gradual change in populations.  ### 2. **Survival of the Fittest**:      He
    emphasized that individuals who possess advantageous traits (e.g., physical strength,
    intelligence) have higher chances of survival and reproduction, which in turn promotes
    the spre

In [25]:
# Both stage-2 arms start from the SAME model: the anchored stage-1 checkpoint. So this panel
# isolates stage-2 anchoring alone -- it is not an anchored chain against an unanchored one.
STAGE2_PANEL = {
    "after A":    (STAGE1_FUSED, None),
    "unanchored": (STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model"),
    "anchored":   (exported, None),
}

t0 = time.time()
answers_b = run_panel(STAGE2_PANEL, PANEL_B)
print(f"stage-2 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  after A     the anchored stage-1 model, before the cookbook\n"
      f"  unanchored  domain B on top of it with lambda = mu = 0, own best dose ({DOSE_B} epochs)\n"
      f"  anchored    domain B on top of it with the anchor ({EPOCHS} epochs)")
show(answers_b, PANEL_B)

stage-2 panel: 3 models x 5 probes, 51 s
  after A     the anchored stage-1 model, before the cookbook
  unanchored  domain B on top of it with lambda = mu = 0, own best dose (3 epochs)
  anchored    domain B on top of it with the anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [after A]
    Darwin's theory of natural selection is a fundamental principle in evolutionary biology.
    He explained that all living beings have been created through the gradual process of
    modification and inheritance, which he called "natural selection." Here are some key
    points:  1. **Gradual Change**: Natural selection acts over long periods (millions to
    millions of years) on inherited variations. 2. **Inheritance of Variations**: These
    variations become more common or advantageous as they persist in successive generations.
    3. **Direction of Change**: The changes tend to be directed toward improving existing
    forms. 4. **Natural Selection Produces New Species**: 

### What the probes showed, including what they did not

**The decoder first, because it changes how the rest reads.** An earlier decode of these same ten
probes, at the same 200 greedy tokens but with no repetition controls, had *every* trained arm
looping: the stage-2 anchored model answered `B1` by repeating "Put a pound of butter in a large
cask, and pour in the suet pudding, and stir it well" five times and starting a sixth. None of
that survives the settings above. Across all thirty generations exactly one contains a repeated
span of five words or longer, and it is a topic phrase — "the inhabitants of an island", three
times inside one long answer — not a loop. **The degenerate output was the decoder.** Everything
below is what the models do once it is taken away.

**What the controls cost, stated before it is used.** `G2` is the receipt. Under plain greedy
decoding all three stage-1 models opened with "Three prime numbers greater than 10 are"; with the
repetition penalty on, the base model says "greater than **2**" and both trained models say
"greater than **9**". The penalty down-weights a token the prompt already used, and the base model
pays for it exactly as the others do. All three still list three correct primes above 10. This is
the price of a readable panel, and it is charged to every model equally.

**Adaptation is visible, but not on both arms equally.** The unanchored stage-1 control's
adaptation is total and unmistakable: it answers all five probes — including both general ones —
in Darwin's own first person and in the book's argumentative style. The anchored model's is much
harder to see in the text. It uses the domain's vocabulary on `A1` and `A2` ("struggle for
existence", "variations", "modification") but keeps the base model's register: numbered points and
a closing summary. Shown only its stage-1 answers, you would struggle to say what corpus it had
been trained on. That is a real limitation of this panel, and it points at the table: section 8b
puts the anchored run at 17.45 on Darwin against the base model's 30.12, so the adaptation is
there — it is just not what the prose style shows you.

**On the general axis the control's damage is now legible, and it was not before.** This is the
biggest change from the looping decode, where `G1` and `G2` looked like ties and the section had to
say that nothing catastrophic had happened and no more. Asked what photosynthesis is, the stage-1
control opens correctly and then drifts into invented Victorian biology — "the greater the amount
of oxygen produced, the more rapidly the cells will divide and become larger" — for the rest of its
budget. Asked on `I1` for two sentences for a ten-year-old, it produces a passage about island
faunas containing a fabricated statistic ("about 1000 fish in the sea, compared with only 250 in
the open ocean over the whole world"). The anchored model answers `I1` in exactly two plain
sentences, using natural selection as the explanation — domain content delivered inside behaviour
the base model already had, which is what `I*` was written to look for. Section 8b's WikiText-2
column had said the same thing first: 18.87 for that control against 16.09 anchored, base 17.80.

**Stage 2 splits, and both halves agree with the table.** Retention went the way the method
predicts and by a wide margin. Asked about natural selection after learning the cookbook, the
unanchored arm answers about Darwin's *biography* and invents a quotation for him — "I have known
many persons who were born with disabilities, but succeeded in their own ways" — while the anchored
arm stays on the concept ("he said that these traits were not accidental but had been selected from
other forms of life"). A · Darwin: 27.02 for the control, 18.92 anchored.

The new domain went the other way, and just as plainly. On `B1` the control writes the fullest
period recipe — "Take three quarts of milk, two pounds and a half of sugar, one pound of butter,
and the same of lard; put them in a dutch-oven" — while the anchored model's is in the same voice
but vaguer and more confused, opening "Put together the ingredients, and put them in a large pot"
and later calling for "stirring-pieces of nutmegs". On `B2` the control gives a real 1853 method;
the anchored model, asked how to **roast**, instructs you to boil enough water to cover the meat
and roast it in that. Both answers are coherent — that part was the decoder — but on the domain it
just learned the control is better, which is what B · cookery 12.99 against 15.13 said first. **The
anchor makes that trade; it does not abolish it.**

The remaining probes are ties. `G1` in the second panel: all three give two correct sentences.
`I2`: both trained models give a recipe and the one-sentence explanation asked for, and the only
model that fails is the one that has *not* seen domain B — it announces "**A simple soup recipe:**"
and then describes boiling rather than giving a recipe. `A2` is a tie of two different kinds: the
control argues the book's case in the book's voice, the anchored model answers the question in the
base model's voice with the domain's terms, and nothing here can rank those.

**The count.** By the narrow criterion of answering the question asked, in the form asked, without
inventing: the anchored model is better on four probes (`A1`, `G1` and `I1` in the first panel,
`A1` in the second), worse on two (`B1`, `B2`), and indistinguishable on the remaining four. It
does not adapt *better* on either domain at either stage; on both it fits the corpus less well than
the control does, and the tables said so first.

**Where the same model appears in both panels its answers are identical** — stage 1's `anchored`
and stage 2's `after A` are the same checkpoint, and their `A1` and `G1` answers match character
for character. Decoding is deterministic, so re-running this section reproduces it.

**And the standing caveat, which the cleaner output does not weaken.** Thirty single greedy
generations from one 0.6B model are an illustration, not a measurement. Nothing here is scored by
anything, no claim in this notebook rests on it, and a different set of ten questions could tell a
different story. The perplexity tables are the evidence; this is what the evidence sounds like.


## 12. Integration, and what it should do in both directions

Everything up to here measures how much the model **keeps**. That framing makes the anchor sound
like a brake — something that stops training before it goes too far — and a brake is not very
interesting. The mechanism says something stronger than "less is lost", and this section states
what it says and then puts it to the models to see how much of it actually shows at this scale.

### What the anchor does, mechanically

One term does the work. For each sub-module of the network — an attention projection, an MLP
block — the training loss carries

> `λ · E` over `h ~ p(h)` of `‖ f_student(h) − f_teacher(h) ‖²`

Read it right to left. `f` is one sub-module, treated as a function. `h` is a *hidden state* — one
of the vectors that flows between the model's layers, which is what a sub-module actually takes as
input. `p(h)` is a distribution over those vectors, estimated from the **base** model on general
text and shipped inside the artifact section 1 downloaded; sampling from it costs no data and no
forward pass over anything the model once saw. So the term draws inputs the base model already
produces, pushes each through one sub-module in both the old model and the one being trained, and
charges for the difference in what comes out.

What that penalises is not weight movement as such — the much smaller `μ` term does that. It
penalises **changing what a piece of the model computes on the inputs it already sees**. A weight
change that leaves every familiar hidden state mapped where it was is free no matter how large it
is; one that re-purposes a sub-module is expensive no matter how small.

That leaves gradient descent an easy route to the new corpus and a hard one. The hard route is to
overwrite — carve out fresh machinery for the cookbook and pay for every familiar input that
machinery no longer handles the way it did. The easy route is to reach the same behaviour
*through* structure that already exists: encode "brown the meat first" using whatever the model
already has for heat, for food, for cause and effect. The anchor cannot tell the two routes apart
and does not try. It just makes the first one expensive. That is the whole of the constraint, and
the phrase for adapting under it is **guided integration**.

### Why a constraint like that predicts traffic in both directions

If new content has to be written through machinery that already exists, then the new content and
the old are not in separate compartments — they are expressed in the same representations, so each
should be reachable from the other. Two predictions follow, and they are not the same prediction:

- **Inward — prior knowledge reaching the new domain.** Ask, about the corpus's own subject,
  something the corpus does not contain. Every corpus is a slice of a subject, never the whole of
  it: *The Origin of Species* (1859) predates genetics and germ theory, and an 1853 cookbook gives
  method without chemistry. A model that has integrated the corpus should still be able to bring
  what it already knew to those questions.
- **Outward — domain semantics reaching adjacent questions.** Ask something the corpus is *not*
  about but which shares its concepts. The corpus should show up in the answer, because that is
  where the new content was written.

### Outward transfer is only a virtue if it is selective

This is the part that needs the control, and section 11 has already shown why. The unanchored
stage-1 model transfers the domain **everywhere**: asked what photosynthesis is, it answers
correctly for one sentence and then drifts into invented Victorian biology; asked for two sentences
for a ten-year-old, it returns a passage about island faunas with a fabricated statistic. Domain
semantics arriving in a question that has nothing to do with the domain is not integration — it is
the domain having eaten the model, and it is the ordinary failure mode of fine-tuning a small model
on a small corpus.

So transfer on its own proves nothing, and a panel that only asked adjacent questions would be
unreadable. Each adjacent question below is therefore paired with an **unrelated** one, and the
demonstration is the *difference between the two*, not either alone. A model that draws on the
corpus for the adjacent question and leaves it out of the unrelated one has done something the
model that does it for both has not.

### The standing of this claim, stated before the outputs

The mechanism above is arithmetic: it is what the loss term computes, and that is not in question.
Whether the resulting model is *better integrated* in any measurable sense is a different question
with a worse answer. The research programme behind this package built two structural measures of
integration and neither predicted judged quality — one of them came out with the sign backwards.
So: what follows is eight questions put to six models, every generation printed in full and
uncurated. It illustrates behaviour consistent with the mechanism. It is not evidence for it,
nothing in this notebook rests on it, and a different eight questions could read differently.

| | |
|---|---|
| `P*` | **inward** — about the domain's subject, answerable only from knowledge the corpus does not hold |
| `Q*` | **outward, adjacent** — not about the corpus, but sharing its concepts; domain content here *fits* |
| `N*` | **outward control, unrelated** — shares nothing with either corpus; domain content here is a **fault** |

The eight probes were written down before any output was looked at, and none was reworded, dropped
or reordered afterwards. `Q3` is the one that reads both stages at once: it is adjacent to domain
A, and it is put to models that have since been trained on domain B. `N1` and `N2` are the same
question, so the two panels can be read against each other. Decoding is exactly section 11's —
greedy, 200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`, the chat template with
thinking off — identical for every model and every probe, with nothing tuned per question.

In [ ]:
# The transfer probes, fixed before any output was seen: nothing below was reworded, dropped or
# reordered afterwards. Same three models as section 11, the same `GENERATION` settings and the
# same helpers -- only the questions are new.
#
#   P*  inward   -- knowledge the corpus does NOT contain, asked about the corpus's own subject
#   Q*  outward  -- a question the corpus is not about, but which shares semantics with it
#   N*  control  -- a question sharing nothing with either corpus, where transfer is a FAULT
PANEL_T1 = [
    ("P1", "How does natural selection relate to genes and DNA?"),
    ("P2", "Does natural selection explain why bacteria become resistant to antibiotics?"),
    ("Q1", "Why do species introduced to a new island often multiply rapidly?"),
    ("N1", "What is the capital of Japan, and roughly how many people live there?"),
]

t0 = time.time()
answers_t1 = run_panel(STAGE1_PANEL, PANEL_T1)
print(f"stage-1 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either\n"
      f"  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question\n"
      f"  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode")
show(answers_t1, PANEL_T1)

stage-1 transfer panel: 3 models x 4 probes, 59 s
  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either
  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question
  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode

P1  How does natural selection relate to genes and DNA?

  [base]
    Natural selection is a fundamental mechanism of evolution, but it operates through
    **genetic variation**, not directly through **genes** or **DNA itself**. Here's how they
    relate:  ### 1. **Genes are the basis of heredity**    - Genes (DNA sequences) carry
    instructions for building proteins, which in turn determine traits.    - These genetic
    information is passed from parents to offspring.  ### 2. **Variation occurs due to
    mutations**    - Mutations can introduce new alleles (different versions of a gene),
    leading to genetic diversity.    - This variability is crucial because it allows

In [ ]:
# Stage 2, same three models as the stage-2 panel above -- all starting from the anchored
# stage-1 checkpoint. `Q3` is the one that reads both stages at once: it is adjacent to domain A,
# put to models that have since been trained on domain B.
PANEL_T2 = [
    ("P3", "Why does browning meat before stewing improve the flavour?"),
    ("Q2", "How did households keep food from spoiling before refrigeration?"),
    ("N2", "What is the capital of Japan, and roughly how many people live there?"),
    ("Q3", "In a place with little food, what happens to a population of animals over many "
           "generations?"),
]

t0 = time.time()
answers_t2 = run_panel(STAGE2_PANEL, PANEL_T2)
print(f"stage-2 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works\n"
      f"  Q2 outward  -- adjacent to domain B\n"
      f"  N2 control  -- the same unrelated question as N1, so the two panels can be compared\n"
      f"  Q3 outward  -- adjacent to domain A, asked after domain B was learned")
show(answers_t2, PANEL_T2)

stage-2 transfer panel: 3 models x 4 probes, 55 s
  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works
  Q2 outward  -- adjacent to domain B
  N2 control  -- the same unrelated question as N1, so the two panels can be compared
  Q3 outward  -- adjacent to domain A, asked after domain B was learned

P3  Why does browning meat before stewing improve the flavour?

  [after A]
    The improvement of flavor in meat when browned before stewing is due to several factors:
    1. **Increased Fat and Moisture**: Brown meat contains more fat, moisture, and protein
    than white meat. These substances are better absorbed by water-soluble ingredients like
    starches or oils during cooking.  2. **Improved Texture**: The brown color suggests a
    well-cooked surface with less dryness, which enhances the texture and taste.  3.
    **Enhanced Flavor Profile**: When cooked brownly, the meat has a richer flavor because
    it contains more aromatic compounds (like tannins

### What the panel showed, and which half of the idea did not appear

**Outward transfer is visible, and — the part that matters — it is selective.** Four exhibits;
the first two are a pair and have to be read together.

`N1` is the clearest of the four. The question has nothing to do with Darwin. The unanchored
control answers it correctly in two sentences and then spends its remaining 165 words on Victorian
biogeography: *"The capital of Japan is Tokyo. It has a population of about 10 million
inhabitants. **In South America, we find that the number of inhabitants in each country varies
greatly from one to another**…"*, on through the southern United States, Australia and New Zealand.
The anchored model answers in four sentences about Tokyo's population and its role in trade and
culture, and stops. Both models learned the same corpus; only one of them brings it to a geography question.

`Q1` is the other half of that pair, and it has to be read against `N1` to mean anything. Asked why
species introduced to an island multiply, the **base** model gives six generic factors and never
names natural selection at all — its list runs "Adaptation and Survival", "Genetic Diversity",
"Resource Availability", "Human Intervention". The anchored model leads with it: *"1. **Natural
Selection**: The introduction of new species can lead to natural selection, as the best-adapted
individuals will survive and reproduce more successfully"*, then competition for resources and
altered niches. The domain's central concept arrives on an adjacent question where the base model
did not raise it — and does not arrive on `N1`. The unanchored control also leads with natural
selection on `Q1`; it just does the same thing on `N1`. **The difference between the two arms is
not whether the domain transfers, it is whether the transfer is aimed.**

`Q3` carries the most weight of the four, because it reads both stages at once: a
domain-A-adjacent question put to models that have since been trained on the cookbook. The anchored arm still has
Darwin — *"the population would still diminish by natural selection: for instance, when a new
variety of cattle was introduced into an area which had no such cattle, the old stock would
gradually give way to this new one"*. The unanchored arm has lost the subject completely and
answers in the cookbook's household register: *"A small group of animals in an unimproved state
will soon become sick and weak… It is very important to keep children well fed"*. The string
"natural selection" does not occur in its answer. (One caution on the anchored answer: cattle,
sheep and flocks belong to *both* corpora — Darwin opens on domestic breeds and the cookbook is
full of meat — so the livestock framing cannot be attributed to either.)

`Q2` is a quieter version of the same thing on domain B. Asked how food was kept before
refrigeration, the model that has **not** seen the cookbook invents a history of canning, complete
with *"Some culls (like those made by bees)"* and a claim that canning declined after the 18th
century. The anchored model answers from the period's actual practice — a wooden box or tin case,
*"you can use salt as an ingredient"*, *"a little vinegar in the box"* — which is domain content
arriving where it belongs. It is a mixed answer, with two garbled sentences at the end, and it is
recorded above in full.

**Inward transfer shows at stage 1, and stage 2 cannot test it at this scale.** On `P1` and `P2`
the base model knows genetics and antibiotic resistance well, so there is something to transfer,
and the two trained arms separate sharply. The unanchored model has simply lost the knowledge:
its `P1` answer never contains the string "DNA" at all, opening *"Genes are the basic units of
variation, and they act through inheritance"* before dissolving into *The Origin*'s prose about
varieties in close contact. The anchored model keeps it and frames it in the domain's terms —
*"DNA provides the blueprint, but natural selection determines which traits become dominant"* —
and on `P2` gives horizontal gene transfer and selection pressure, neither of which any 1859 text
can supply.

`P3` fails to test anything, and that is worth saying plainly rather than reading a result into
it. Asked why browning meat improves flavour, **none** of the three models knows the chemistry —
not even "after A", which has never seen a cookbook and is the closest thing here to an untouched
control. It answers that *"Brown meat contains more fat, moisture, and protein than white meat"*,
which confuses browning with brown meat and is wrong. Prior knowledge cannot transfer inward if
the model never had it, and a 0.6B model does not have the Maillard reaction. The probe measures
the model's size, not the method.

**What went against the anchored arm, uncurated.** `N2` is the anchored model's worst answer in the
section: it gives Tokyo correctly in two sentences, then invents a question of its own —
*"What are some natural wonders in Europe?"* — and lists thirty of them, including the Welsh Wastes,
the Spanish Andes and the Polish Poles. By the reading rule set out above it still *passes* `N2`:
no cookery vocabulary appears, so nothing transferred where it should not. But it is a bad answer,
the unanchored control's thirteen words are the best one on that probe, and the selectivity result
at stage 2 rests on the absence of a fault rather than on quality. Two smaller ones: the **base**
model answers `N1` with "Osaka City", so the control question cannot be read as the trained models
degrading — at this size the base model is not a gold standard either. And the anchored `P1` answer
opens with a claim that is arguably wrong ("operates on the variation of individuals, not on their
genes or DNA") where the base model is more careful.

**The summary, direction by direction.** Outward transfer and its selectivity: **visible**, at both
stages, most convincingly on `N1`/`Q1` as a pair and on `Q3`. Inward transfer: **visible at stage
1**, on `P1` and `P2`, where it takes the form of the anchored model retaining knowledge the
control has lost and answering with it inside the domain's framing; **not testable at stage 2**,
because the prior knowledge `P3` asks for is not in a model this small. And the standing caveat is
unchanged by any of it — twenty-four greedy generations are an illustration of a mechanism, not a
measurement of one, and the tables above are still where the evidence in this notebook lives.

## What this showed, and what it did not

**Showed.** Two domains adapted in sequence on one 0.6B model, with the second stage reading
none of the first domain's text and the workspace storing none of it. Both stages measured on
two axes against a control that differs only in λ and μ, and both of those controls
measured again at their own best dose, because a control run past its optimum flatters the
thing it is controlling for.

**Did not show.** Anything statistical. One seed, one run per cell, corpora roughly a quarter of
what the operating point was tuned on by bytes, a quarter of its epochs, and no repetition — so a
difference of a few percent in any cell is not distinguishable from run-to-run variation here.
It also did not reproduce any published number, and does not try to: the paper's own headline
is its measurement on its corpus and its instruments.

**If you want to push it further**, the honest next steps are the boring ones: run each cell at
a second seed, put the shipped 15 epochs on a corpus the size they were tuned for, and re-tune
λ if you change the rank or the artifact — it is coupled to both, and a ported λ measures the
port rather than the method.

### Where to go next

| | |
|---|---|
| `docs/concepts.md` | what the anchor does, what λ and μ are, how a run is read |
| `docs/recipes.md` | the shipped operating point field by field, and its couplings |
| `docs/multi-domain-chains.md` | a third domain, `lfa chain`, and what the paper found after one |
| `docs/faq.md` | GPU memory, full weights, reading the general axis |
| `examples/quickstart.py` | the single-domain flow as a script |